# Feature Engineering
**CAP182 Capstone Project: STADIOEquities**

Input: `data/cleaned_data.csv` (created by `preprocessing/preprocessing.ipynb`).
Outputs: `data/train.csv`, `data/test.csv` and `models/preprocessor.pkl`, used by both model notebooks.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
os.makedirs("../models", exist_ok=True)

data = pd.read_csv("../data/cleaned_data.csv")
print(data.shape)
data.head()

(41176, 20)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 1. Replace the `pdays` placeholder with a binary flag
`pdays = 999` is a code meaning "never previously contacted", not a real number of days. It is replaced by `previously_contacted` (1 = contacted in an earlier campaign, 0 = not).

In [2]:
data["previously_contacted"] = (data["pdays"] != 999).astype(int)
data = data.drop(columns=["pdays"])
data["previously_contacted"].value_counts()

previously_contacted
0    39661
1     1515
Name: count, dtype: int64

## 2. Encode the target
`yes` becomes 1 and `no` becomes 0.

In [3]:
data["y"] = (data["y"] == "yes").astype(int)
data["y"].value_counts()

y
0    36537
1     4639
Name: count, dtype: int64

## 3. Time-ordered train/test split
Rows are in date order (May 2008 to November 2010). The first 80% (earlier campaigns) are used for training and the last 20% (later campaigns) for testing. `shuffle=False` keeps the chronological order (Moro, Cortez and Rita, 2014).

In [4]:
X = data.drop(columns="y")
y = data["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

pd.DataFrame({
    "Rows": [len(y_train), len(y_test)],
    "Conversion rate": [round(y_train.mean(), 3), round(y_test.mean(), 3)],
}, index=["Train (earlier)", "Test (later)"])

,Rows,Conversion rate
Train (earlier),32940,0.064
Test (later),8236,0.308


## 4. Check whether test-period values fall inside the training range

In [5]:
macro = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
ranges = pd.concat({"Train": X_train[macro].agg(["min", "max"]).T,
                    "Test": X_test[macro].agg(["min", "max"]).T}, axis=1)
print(ranges)
print("\nMonths in test but not in train:", sorted(set(X_test["month"]) - set(X_train["month"])))

                   Train                Test          
                     min       max       min       max
emp.var.rate      -1.800     1.400    -3.400    -1.100
cons.price.idx    92.756    94.465    92.201    94.767
cons.conf.idx    -50.000   -36.100   -50.800   -26.900
euribor3m          1.299     5.045     0.634     1.299
nr.employed     5099.100  5228.100  4963.600  5099.100

Months in test but not in train: ['sep']


## 5. Encoding and scaling
- **One-hot encoding** turns each category into its own 0/1 column. `handle_unknown="ignore"` handles categories that appear only in the test period (September).
- **Standard scaling** gives each numerical feature a mean of 0 and a standard deviation of 1, so features with large units (such as `nr.employed`) do not dominate logistic regression.

The transformer is saved **unfitted**. Each model notebook places it inside a pipeline, so it is fitted on training data only, which prevents leakage from the test set.

In [6]:
cat_cols = ["job", "marital", "education", "default", "housing", "loan",
            "contact", "month", "day_of_week", "poutcome"]
num_cols = ["age", "campaign", "previous", "emp.var.rate", "cons.price.idx",
            "cons.conf.idx", "euribor3m", "nr.employed", "previously_contacted"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

# Check the output shape (fitted on training data only)
print("Train after encoding:", preprocessor.fit_transform(X_train).shape)
print("Test after encoding: ", preprocessor.transform(X_test).shape)

Train after encoding: (32940, 61)
Test after encoding:  (8236, 61)


## 6. Save outputs

In [7]:
train = X_train.assign(y=y_train)
test = X_test.assign(y=y_test)
train.to_csv("../data/train.csv", index=False)
test.to_csv("../data/test.csv", index=False)

from sklearn.base import clone
joblib.dump(clone(preprocessor), "../models/preprocessor.pkl")   # unfitted copy
print("Saved ../data/train.csv, ../data/test.csv and ../models/preprocessor.pkl")

Saved ../data/train.csv, ../data/test.csv and ../models/preprocessor.pkl
